<a href="https://colab.research.google.com/github/adg1205/CSE425-Project/blob/master/Easy%20Task/Easy_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!sudo apt update -qq

54 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
!sudo apt install -y ffmpeg -qq

ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 54 not upgraded.


**Easy Task Implementation**

In [4]:
AUDIO_ROOT = "/content/drive/MyDrive/Project Dataset/genres_original"     # <-- your path
OUT_DIR    = "/content/drive/MyDrive/easy_task_results"   # change if you want

import os
os.makedirs(OUT_DIR, exist_ok=True)
LATVIS_DIR = os.path.join(OUT_DIR, "latent_visualization")
os.makedirs(LATVIS_DIR, exist_ok=True)

print("AUDIO_ROOT exists?", os.path.isdir(AUDIO_ROOT), AUDIO_ROOT)

AUDIO_ROOT exists? True /content/drive/MyDrive/Project Dataset/genres_original


In [5]:
!pip -q install librosa soundfile scikit-learn umap-learn matplotlib tqdm

In [6]:
import os, glob, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import librosa
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score  # required metrics [file:1]
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# -------- data indexing --------
def build_index(audio_root):
    genre_dirs = sorted([d for d in glob.glob(os.path.join(audio_root, "*")) if os.path.isdir(d)])
    rows = []
    for gdir in genre_dirs:
        genre = os.path.basename(gdir)
        wavs = sorted(glob.glob(os.path.join(gdir, "*.wav")))
        for wav in wavs:
            track_id = os.path.splitext(os.path.basename(wav))[0]
            rows.append((genre, track_id, wav))
    return pd.DataFrame(rows, columns=["genre", "track_id", "path"])

# -------- feature extraction (simple) --------
def mfcc_vector(wav_path, sr=22050, duration=30.0, n_mfcc=20):
    y, _ = librosa.load(wav_path, sr=sr, mono=True, duration=duration)
    if y is None or len(y) == 0:
        raise ValueError("Empty audio")
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)  # (n_mfcc, T)
    feat = np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)], axis=0)  # (2*n_mfcc,)
    return feat.astype(np.float32)

class FeatureDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i]

# -------- basic VAE (Easy requirement) --------
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=8, hidden_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        self.fc2 = nn.Linear(latent_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        h = torch.relu(self.fc1(x))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = torch.relu(self.fc2(z))
        return self.fc_out(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar

def vae_loss(x, x_hat, mu, logvar, beta=1.0):
    recon = F.mse_loss(x_hat, x, reduction="mean")
    kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + beta * kld

# -------- 1) Load + featurize --------
df = build_index(AUDIO_ROOT)
print("Tracks:", len(df), "| genres:", df["genre"].nunique())

X_list, keep = [], []
for _, r in tqdm(df.iterrows(), total=len(df)):
    try:
        X_list.append(mfcc_vector(r["path"]))
        keep.append(True)
    except Exception:
        keep.append(False)

df_ok = df[keep].reset_index(drop=True)
X = np.stack(X_list, axis=0)

# standardize
scaler = StandardScaler()
Xz = scaler.fit_transform(X).astype(np.float32)

# cluster count: using number of genre folders as a reasonable k for this dataset
k = df_ok["genre"].nunique()

# -------- 2) Train VAE --------
vae = VAE(input_dim=Xz.shape[1], latent_dim=8, hidden_dim=128).to(DEVICE)
opt = torch.optim.Adam(vae.parameters(), lr=1e-3)
loader = DataLoader(FeatureDataset(Xz), batch_size=128, shuffle=True)

vae.train()
for ep in range(1, 51):
    for xb in loader:
        xb = xb.to(DEVICE)
        x_hat, mu, logvar = vae(xb)
        loss = vae_loss(xb, x_hat, mu, logvar, beta=1.0)
        opt.zero_grad()
        loss.backward()
        opt.step()
    if ep % 10 == 0 or ep == 1:
        print(f"Epoch {ep:02d} loss={float(loss.detach().cpu()):.4f}")

# -------- 3) VAE latent -> KMeans --------
vae.eval()
with torch.no_grad():
    mu, _ = vae.encode(torch.tensor(Xz).to(DEVICE))
Z = mu.cpu().numpy()

c_vae = KMeans(n_clusters=k, random_state=SEED, n_init="auto").fit_predict(Z)
sil_vae = silhouette_score(Z, c_vae)
ch_vae  = calinski_harabasz_score(Z, c_vae)

# -------- 4) Baseline: PCA -> KMeans --------
Xp = PCA(n_components=8, random_state=SEED).fit_transform(Xz)
c_pca = KMeans(n_clusters=k, random_state=SEED, n_init="auto").fit_predict(Xp)
sil_pca = silhouette_score(Xp, c_pca)
ch_pca  = calinski_harabasz_score(Xp, c_pca)

# -------- 5) Visualize t-SNE (Easy requirement: t-SNE or UMAP) --------
Z2 = TSNE(n_components=2, random_state=SEED, init="pca", learning_rate="auto").fit_transform(Z)
plt.figure(figsize=(7,6))
plt.scatter(Z2[:,0], Z2[:,1], c=c_vae, s=10, cmap="tab10")
plt.title("t-SNE of VAE latent (KMeans clusters)")
plt.tight_layout()
plt.savefig(os.path.join(LATVIS_DIR, "tsne_vae_kmeans.png"), dpi=200)
plt.close()

Xp2 = TSNE(n_components=2, random_state=SEED, init="pca", learning_rate="auto").fit_transform(Xp)
plt.figure(figsize=(7,6))
plt.scatter(Xp2[:,0], Xp2[:,1], c=c_pca, s=10, cmap="tab10")
plt.title("t-SNE of PCA(8) (KMeans clusters)")
plt.tight_layout()
plt.savefig(os.path.join(LATVIS_DIR, "tsne_pca_kmeans.png"), dpi=200)
plt.close()

# -------- 6) Save metrics CSV (deliverable artifact) --------
metrics = pd.DataFrame([
    {"method": "VAE(mu)+KMeans", "k": k, "embed_dim": Z.shape[1],
     "silhouette": sil_vae, "calinski_harabasz": ch_vae},
    {"method": "PCA(8)+KMeans", "k": k, "embed_dim": Xp.shape[1],
     "silhouette": sil_pca, "calinski_harabasz": ch_pca},
])
metrics_path = os.path.join(OUT_DIR, "clustering_metrics.csv")
metrics.to_csv(metrics_path, index=False)

print("Saved:", metrics_path)
print("Saved plots in:", LATVIS_DIR)
metrics

DEVICE: cuda
Tracks: 700 | genres: 7


100%|██████████| 700/700 [05:16<00:00,  2.21it/s]


Epoch 01 loss=0.9893
Epoch 10 loss=0.9493
Epoch 20 loss=0.8248
Epoch 30 loss=0.7816
Epoch 40 loss=0.7198
Epoch 50 loss=0.7191
Saved: /content/drive/MyDrive/easy_task_results/clustering_metrics.csv
Saved plots in: /content/drive/MyDrive/easy_task_results/latent_visualization


,method,k,embed_dim,silhouette,calinski_harabasz
0,VAE(mu)+KMeans,7,8,0.324799,372.262421
1,PCA(8)+KMeans,7,8,0.190749,199.359238


In [7]:
import os
import umap
import matplotlib.pyplot as plt

umap_2d = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
).fit_transform(Z)  # fit_transform is the standard usage for getting an embedding [web:144]

plt.figure(figsize=(7, 6))
plt.scatter(umap_2d[:, 0], umap_2d[:, 1], c=c_vae, s=10, cmap="tab10")
plt.title("UMAP of VAE latent (KMeans clusters)")
plt.tight_layout()
plt.savefig(os.path.join(LATVIS_DIR, "umap_vae_kmeans.png"), dpi=200)
plt.close()

umap_2d_pca = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
).fit_transform(Xp)  # same UMAP pattern [web:144]

plt.figure(figsize=(7, 6))
plt.scatter(umap_2d_pca[:, 0], umap_2d_pca[:, 1], c=c_pca, s=10, cmap="tab10")
plt.title("UMAP of PCA(8) (KMeans clusters)")
plt.tight_layout()
plt.savefig(os.path.join(LATVIS_DIR, "umap_pca_kmeans.png"), dpi=200)
plt.close()

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
